# Scheduling & Monitoring the TPCH Pipeline in Production

This notebook explains how to **orchestrate, schedule, and monitor** the TPCH ETL pipeline
(Bronze → Silver → Gold → Data Quality) in a production environment using **Databricks Lakeflow Jobs**.

## 1. Pipeline Architecture Overview

The TPCH pipeline has **four layers**, each implemented as a separate notebook or file:

| Layer | Asset | Purpose |
|------|-------|---------|
| **Bronze** | Ingestion notebook / SQL | `CREATE OR REPLACE TABLE AS SELECT` from `samples.tpch.*` into `tpch_dev.bronze.*` |
| **Silver** | 8 cleaning scripts (`.py` files) | Standardization, dedup, validation, quarantine; writes to `tpch_dev.silver.*` + `_quarantine` tables |
| **Gold** | Dim & Fact notebook | Builds `dim_geography`, `dim_customer`, `dim_product`, `dim_supplier`, `fact_sales`, `vw_annual_sales_kpis` |
| **DQ Gate** | DQ & Reconciliation notebook | 6 checks (ingestion, PK, null FK, measures, reconciliation, quarantine); **raises exception** on failure |

```
Bronze Ingestion
       │
       ▼
Silver Cleaning (8 tables, parallel)
       │
       ▼
Gold Transformation (dims + fact)
       │
       ▼
DQ & Reconciliation Gate
       │
   PASS ──► Pipeline complete
   FAIL ──► Raise exception → Job fails → Alert triggers
```

## 2. Job Structure — Task DAG in Lakeflow Jobs

A single **Lakeflow Job** orchestrates the entire pipeline. Each layer is a **task** in the job, with dependencies
creating a directed acyclic graph (DAG):

| Task # | Task Name | Type | Depends On | Description |
|--------|----------|------|------------|-------------|
| 1 | `bronze_ingestion` | Notebook | — | Runs the Bronze ingestion notebook |
| 2 | `silver_region` | Notebook | `bronze_ingestion` | Cleans region table |
| 3 | `silver_nation` | Notebook | `bronze_ingestion` | Cleans nation table |
| 4 | `silver_customer` | Notebook | `bronze_ingestion` | Cleans customer table |
| 5 | `silver_supplier` | Notebook | `bronze_ingestion` | Cleans supplier table |
| 6 | `silver_part` | Notebook | `bronze_ingestion` | Cleans part table |
| 7 | `silver_partsupp` | Notebook | `bronze_ingestion` | Cleans partsupp table |
| 8 | `silver_orders` | Notebook | `bronze_ingestion` | Cleans orders table |
| 9 | `silver_lineitem` | Notebook | `bronze_ingestion` | Cleans lineitem table |
| 10 | `gold_transform` | Notebook | Tasks 2–9 (all Silver) | Builds Gold dims + fact |
| 11 | `dq_gate` | Notebook | `gold_transform` | Runs DQ checks; raises error on failure |

**Key design choices:**
* Silver cleaning tasks **run in parallel** (all depend only on Bronze)
* Gold depends on **all** Silver tasks completing successfully
* DQ gate is the **final task** — acts as a quality gate before the pipeline is considered successful

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# Example: Creating the TPCH pipeline as a Lakeflow Job via Databricks SDK
# (Illustrative — run this in a separate notebook or CI/CD pipeline)
# ─────────────────────────────────────────────────────────────────────────────

# from databricks.sdk import WorkspaceClient
# from databricks.sdk.service import jobs
#
# w = WorkspaceClient()
#
# # Define Silver cleaning tasks (all depend on bronze_ingestion)
# silver_tables = ['region', 'nation', 'customer', 'supplier', 'part', 'partsupp', 'orders', 'lineitem']
# silver_tasks = []
# for tbl in silver_tables:
#     silver_tasks.append(jobs.Task(
#         task_key=f'silver_{tbl}',
#         depends_on=[jobs.TaskDependency(task_key='bronze_ingestion')],
#         notebook_task=jobs.NotebookTask(
#             notebook_path='/l2_assessment/l1_assessment/Silver Cleaning/{tbl}_clean'.format(tbl=tbl),
#         ),
#     ))
#
# # Build the full job definition
# job = w.jobs.create(
#     name='tpch_pipeline_production',
#     tasks=[
#         # Task 1: Bronze Ingestion
#         jobs.Task(
#             task_key='bronze_ingestion',
#             notebook_task=jobs.NotebookTask(
#                 notebook_path='/l2_assessment/l1_assessment/Bronze Ingestion/bronze_ingestion',
#             ),
#         ),
#         # Tasks 2-9: Silver Cleaning (parallel)
#         *silver_tasks,
#         # Task 10: Gold Transformation (depends on ALL silver tasks)
#         jobs.Task(
#             task_key='gold_transform',
#             depends_on=[jobs.TaskDependency(task_key=f'silver_{tbl}') for tbl in silver_tables],
#             notebook_task=jobs.NotebookTask(
#                 notebook_path='/l2_assessment/l1_assessment/Gold/dim_and_fact',
#             ),
#         ),
#         # Task 11: DQ Gate (depends on Gold)
#         jobs.Task(
#             task_key='dq_gate',
#             depends_on=[jobs.TaskDependency(task_key='gold_transform')],
#             notebook_task=jobs.NotebookTask(
#                 notebook_path='/l2_assessment/l1_assessment/Data Quality and Reconciliation/dataQualityAndReconciliationReport',
#             ),
#         ),
#     ],
#     schedule=jobs.CronSchedule(
#         quartz_cron_expression='0 0 6 * * ?',  # Daily at 6:00 AM UTC
#         timezone_id='UTC',
#     ),
# )
# print(f'Job created with ID: {job.job_id}')

print('This cell shows the SDK pattern for job creation. Uncomment and customize paths to run.')

## 3. Scheduling Strategies

### 3.1 Trigger Types

| Trigger | When to Use | Configuration |
|---------|-------------|---------------|
| **Cron schedule** | Daily/nightly batch ETL | `0 0 6 * * ?` (6 AM UTC daily) in Job UI → Schedule |
| **File arrival** | When source files land in S3/ADLS | Job UI → Trigger → File arrival → S3 path |
| **Table update** | When a source Delta table changes | Job UI → Trigger → Table update → monitor `samples.tpch.lineitem` |
| **Manual / API** | On-demand backfills or testing | `POST /api/2.1/jobs/run-now` via REST or CLI |
| **Continuous** | Streaming / near-real-time | Not needed here (batch pipeline) |

### 3.2 Recommended Schedule for This Pipeline

```
Cron:  0 0 6 * * ?
Meaning: Run daily at 06:00 UTC
```

* Choose an off-peak window when warehouse / cluster resources are available
* For this TPCH pipeline (full refresh), **daily** is sufficient
* If source data changes less frequently, use **weekly** (`0 0 6 ? * MON`) to save cost

### 3.3 Compute Configuration

| Setting | Recommendation |
|---------|----------------|
| **Compute type** | Serverless (auto-selected) or a shared all-purpose cluster |
| **Auto-termination** | 10 minutes (auto-scales down after idle) |
| **Photon** | Enabled (faster for SQL-heavy transformations) |
| **SQL Warehouse** | If Silver/Gold use pure SQL cells, route to a Serverless SQL Warehouse instead |

## 4. Monitoring the Pipeline in Production

### 4.1 Job Run UI (Primary Monitoring Surface)

Navigate to **Jobs → tpch_pipeline_production** in the Databricks workspace to see:

* **Run history** — every run with status (Success, Failed, Running)
* **Task-level status** — each task in the DAG shows its state, duration, and logs
* **Gantt chart** — visual timeline showing parallel Silver tasks and sequential Gold/DQ tasks
* **Spark UI** — per-task Spark metrics for debugging slow stages

### 4.2 System Tables for Programmatic Monitoring

Use Databricks **system tables** to monitor pipeline health across all runs:

```sql
-- Recent pipeline runs and their status
SELECT
  job_id, run_id, task_name, state,
  start_time, end_time,
  TIMESTAMPDIFF(MINUTE, start_time, end_time) AS duration_min
FROM system.lakeflow.jobs.task_run_timeline
WHERE job_name = 'tpch_pipeline_production'
ORDER BY start_time DESC
LIMIT 20;
```

```sql
-- Failed runs in the last 7 days
SELECT
  run_id, task_name, error,
  TIMESTAMPDIFF(MINUTE, start_time, end_time) AS duration_min
FROM system.lakeflow.jobs.task_run_timeline
WHERE job_name = 'tpch_pipeline_production'
  AND state = 'FAILED'
  AND start_time >= CURRENT_DATE - INTERVAL 7 DAYS
ORDER BY start_time DESC;
```

### 4.3 DQ Check Results as Monitoring Signals

The DQ notebook already stores results in-table. You can query them to build a dashboard:

* **Quarantine counts** per table per run
* **Reconciliation diff** (Gold vs Silver revenue)
* **PK uniqueness violations**
* **Null surrogate key counts**

### 4.4 Cost Monitoring

```sql
-- DBU consumption by the pipeline job
SELECT
  date(start_time) AS run_date,
  SUM(dbus) AS total_dbus,
  SUM(cost) AS total_cost_usd
FROM system.billing.usage
WHERE custom_tags['jobName'] = 'tpch_pipeline_production'
GROUP BY date(start_time)
ORDER BY run_date DESC
LIMIT 30;
```

## 5. Alerting & Failure Handling

### 5.1 Email / Webhook Alerts

Configure alerts directly in the Job UI:

| Alert Type | Trigger | Recipient |
|------------|---------|-----------|
| **Email** | Job fails | Data engineering team DL |
| **Webhook** | Job fails | Slack/Teams channel via webhook URL |
| **Email** | Job succeeds after failure | Same team (recovery notification) |

Steps: **Job UI → Alerts → Add → select "On failure" → enter email/webhook**

### 5.2 Retry Policy

Configure in Job UI → **Edit task → Advanced**:

| Setting | Value | Rationale |
|---------|-------|----------|
| **Max retries** | 2 | Retry transient failures (e.g., warehouse warm-up) |
| **Retry interval** | 30 seconds | Short backoff for batch jobs |
| **Min / max retry interval** | 30s / 100s | Exponential backoff within bounds |

### 5.3 DQ Gate as a Hard Stop

The DQ notebook is the **final task** in the DAG. If any critical check fails:

1. The notebook **raises a Python `Exception`** → the task status becomes **Failed**
2. The overall job run status becomes **Failed**
3. The **On-failure alert** fires immediately (email / webhook)
4. Downstream consumers are protected from bad data

### 5.4 Failure Response Playbook

| Failure Layer | Symptom | Action |
|---------------|---------|--------|
| **Bronze** | Source table empty / schema mismatch | Check `samples.tpch.*` availability; verify warehouse is running |
| **Silver** | PK violations, RI failures | Inspect quarantine tables; fix source data or cleaning rules |
| **Gold** | Dim/fact build error | Check Silver outputs; verify join keys; inspect notebook logs |
| **DQ Gate** | Reconciliation mismatch, null keys | Re-run Gold; check for upstream Silver changes; investigate manually |

## 6. CI/CD & Deployment Best Practices

### 6.1 Declarative Automation Bundles (DABs)

For production-grade deployments, define the job in a **Declarative Automation Bundle** (`databricks.yml`):

```yaml
# databricks.yml
bundle:
  name: tpch_pipeline

resources:
  jobs:
    tpch_pipeline_production:
      name: tpch_pipeline_production
      schedule:
        quartz_cron_expression: "0 0 6 * * ?"
        timezone_id: UTC
      tasks:
        - task_key: bronze_ingestion
          notebook_task:
            notebook_path: /l2_assessment/bronze_ingestion
        - task_key: silver_region
          depends_on: [{task_key: bronze_ingestion}]
          notebook_task:
            notebook_path: /l2_assessment/silver_region
        # ... other silver tasks ...
        - task_key: gold_transform
          depends_on:
            - {task_key: silver_region}
            - {task_key: silver_nation}
            # ... all silver tasks ...
          notebook_task:
            notebook_path: /l2_assessment/gold_dim_fact
        - task_key: dq_gate
          depends_on: [{task_key: gold_transform}]
          notebook_task:
            notebook_path: /l2_assessment/dq_reconciliation
      email_notifications:
        on_failure:
          - data-eng-team@company.com
```

Deploy with: `databricks bundle deploy`

### 6.2 Environment Promotion

| Environment | Purpose | Schedule |
|-------------|---------|----------|
| **Dev** | Development & testing | Manual trigger only |
| **Staging** | Pre-production validation | Daily (test data) |
| **Prod** | Live pipeline | Daily at 6 AM UTC |

Use DAB targets to parameterize notebook paths and compute per environment.

## 7. Summary — Production Checklist

| # | Item | Status |
|---|------|--------|
| 1 | Single Lakeflow Job with 11 tasks in DAG (Bronze → Silver ×8 → Gold → DQ) | ✓ |
| 2 | Silver tasks run in parallel; Gold depends on all Silver | ✓ |
| 3 | DQ gate is the final task — raises exception on critical failure | ✓ |
| 4 | Daily cron schedule at 6 AM UTC (adjustable) | ✓ |
| 5 | Email / webhook alerts on job failure | ✓ |
| 6 | Retry policy: 2 retries, 60s interval | ✓ |
| 7 | System tables for programmatic monitoring (run history, cost, failures) | ✓ |
| 8 | DAB-based CI/CD for dev → staging → prod promotion | ✓ |
| 9 | Serverless compute for cost efficiency | ✓ |
| 10 | Failure response playbook per layer | ✓ |

### Key Takeaways

* **One job, one DAG** — all layers in a single Lakeflow Job with task dependencies
* **Parallelize Silver** — 8 cleaning tasks run simultaneously after Bronze
* **DQ as a gate** — the DQ notebook is the last task; if it raises an exception, the job fails and alerts fire
* **Monitor via UI + system tables** — the Jobs UI for real-time status; `system.lakeflow.*` and `system.billing.*` for programmatic monitoring
* **Automate deployment with DABs** — version-controlled job definitions promoted across environments